# JAX Performance on CPU and GPU
<a href="https://colab.research.google.com/github/milocortes/ciencia_datos_avanzada/blob/main/notebooks/jax_performance_cpu_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Verifica versión de CUDA instalada

In [ ]:
!nvcc --version

## Instala versión de JAX compatible con la versión de CUDA instalada

In [ ]:
## Si estás usando conda o venv, usa pip para instalar jax
!pip install -U "jax[cuda12]"

In [ ]:
## Si estás usando uv, ejecuta la siguiente instrucción para instalar jax
!uv add "jax[cuda12]"

In [ ]:
import numpy as np
import jax.numpy as jnp 
from jax import device_put, devices, jit

# a function with some amount of calculations 
def f(x):
    y1 = x + x*x + 3
    y2 = x*x + x*x.T
    return y1*y2

# generate some random data
x = np.random.randn(3_000, 3_000).astype('float32')
jax_x_gpu = device_put(jnp.array(x), devices("gpu")[0])
jax_x_cpu = device_put(jnp.array(x), devices("cpu")[0])

# compile function to CPU and GPU backends with JAX
jax_f_gpu = jit(f, backend = "gpu")
jax_f_cpu = jit(f, backend = "cpu")

# warm-up
jax_f_cpu(jax_x_cpu)
jax_f_gpu(jax_x_gpu)

In [ ]:
# Pure NumPy implementation
%timeit -n10 f(x)

In [ ]:
# JAX on CPU
%timeit -n10 f(jax_x_cpu).block_until_ready()

In [ ]:
# JAX JIT-compiled CPU version
%timeit -n10 jax_f_cpu(jax_x_cpu).block_until_ready()

In [ ]:
# JAX on GPU
%timeit -n100 f(jax_x_gpu).block_until_ready()

In [ ]:
# JAX JIT-compiled GPU version
%timeit -n100 jax_f_gpu(jax_x_gpu).block_until_ready()